<a href="https://colab.research.google.com/github/mariya7040ai/-arabic-english-medqa-uncertainty/blob/dev/Paper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Install and imports
!pip -q install -U bitsandbytes accelerate transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 67.8 MB/s eta 0:00:00


In [2]:
# 1b. CONFIG: edit only this cell
LANG = "en"                                   # "en" = English (MedMCQA), "ar" = Arabic (MedAraBench)
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"       # later try the Arabic-focused model, for example inceptionai/jais-adapted-7b-chat
MEDARA_CSV = "/content/medarabench_test.csv"  # only used when LANG = "ar"
N_TEST = 50                                   # Day 2 test size
#N_FULL = 1000                                 # Day 3 full-run size
SEED = 42

In [6]:
# 1b. CONFIG: edit only this cell
LANG = "en"                                   # "en" = English (MedMCQA), "ar" = Arabic (MedAraBench)
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"       # later try the Arabic-focused model, for example inceptionai/jais-adapted-7b-chat
MEDARA_CSV = "/content/medarabench_test.csv"  # only used when LANG = "ar"
N_TEST = 50                                   # Day 2 test size
N_FULL = 1000                                 # Day 3 full-run size
SEED = 42

In [5]:
# 1b. CONFIG: edit only this cell
LANG = "ar"                                   # "en" = English (MedMCQA), "ar" = Arabic (MedAraBench)
#MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"       # later try the Arabic-focused model, for example inceptionai/jais-adapted-7b-chat
MODEL_NAME = "inceptionai/jais-adapted-7b-chat"
MEDARA_CSV = "/content/medarabench_test.csv"  # only used when LANG = "ar"
N_TEST = 50                                   # Day 2 test size
N_FULL = 1000                                 # Day 3 full-run size
SEED = 42

In [7]:
# 2. Seeds and logging
import os, json, time, random, datetime
import numpy as np, pandas as pd, torch

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

LETTERS = list("ABCDEF")

def log_run(text, path="RUN_LOG.md"):
    ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
    with open(path, "a", encoding="utf-8") as f:
        f.write(f"- {ts}: {text}\n")
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

torch 2.11.0+cu130 | cuda: True


In [8]:
# 3. Functions to load MedAraBench (columns as in your Day 1 output)
def load_medarabench(path):
    df = pd.read_csv(path)
    opt_cols = [f"Option {l}" for l in LETTERS]
    rows, bad = [], {"empty_question": 0, "answer_not_a_letter": 0,
                     "answer_points_to_empty_option": 0, "fewer_than_2_options": 0}
    for i, r in df.iterrows():
        q = r["Question"]
        if not isinstance(q, str) or not q.strip():
            bad["empty_question"] += 1; continue
        opts, letters = [], []
        for l, c in zip(LETTERS, opt_cols):
            v = r[c]
            if isinstance(v, str) and v.strip():
                opts.append(v.strip()); letters.append(l)
        ans = r["Correct Answer"]
        if not isinstance(ans, str) or ans.strip().upper() not in LETTERS:
            bad["answer_not_a_letter"] += 1; continue
        ans = ans.strip().upper()
        if ans not in letters:
            bad["answer_points_to_empty_option"] += 1; continue
        if len(opts) < 2:
            bad["fewer_than_2_options"] += 1; continue
        rows.append(dict(id=int(i), question=q.strip(), options=opts,
                         answer_idx=letters.index(ans),
                         specialty=str(r["umbrella_specialty"]), level=str(r["Level"])))
    return pd.DataFrame(rows), bad

In [9]:
# 4. Helpers: convert a table to the common format, and sample questions
def df_to_schema(df, q_col, opt_cols, ans_col, answer_is_index=False, specialty_col=None, level_col=None):
    rows = []
    for i, r in df.iterrows():
        opts = [str(r[c]).strip() for c in opt_cols if isinstance(r[c], str) and str(r[c]).strip()]
        a = r[ans_col]
        idx = int(a) if answer_is_index else LETTERS.index(str(a).strip().upper())
        if len(opts) < 2 or idx >= len(opts):
            continue
        rows.append(dict(id=int(i), question=str(r[q_col]).strip(), options=opts, answer_idx=idx,
                         specialty=str(r[specialty_col]) if specialty_col else "NA",
                         level=str(r[level_col]) if level_col else "NA"))
    return pd.DataFrame(rows)

def sample_df(df, n, seed=SEED):
    return df.sample(n=min(n, len(df)), random_state=seed).reset_index(drop=True)

In [10]:
# 4b. Load the data for the language chosen in CONFIG
if LANG == "ar":
    data_df, bad = load_medarabench(MEDARA_CSV)
    print("dropped:", bad)
else:
    from datasets import load_dataset
    en_raw = load_dataset("openlifescienceai/medmcqa", split="validation").to_pandas()
    print(en_raw.shape, en_raw.columns.tolist())
    print("cop values:\n", en_raw["cop"].value_counts())
    if "choice_type" in en_raw.columns:
        print("choice_type values:\n", en_raw["choice_type"].value_counts())
    spec = "subject_name" if "subject_name" in en_raw.columns else None
    data_df = df_to_schema(en_raw[en_raw["cop"] >= 0], "question", ["opa", "opb", "opc", "opd"], "cop",
                           answer_is_index=True, specialty_col=spec)
K = data_df["options"].map(len)
print(LANG, "usable questions:", len(data_df))
print("options per question:\n", K.value_counts().sort_index())
print("chance accuracy (mean of 1/K): %.3f" % (1 / K).mean())
print("answer position distribution:\n", data_df["answer_idx"].value_counts(normalize=True).sort_index().round(3))
print("specialties:\n", data_df["specialty"].value_counts().head(25))
log_run(f"Loaded {LANG} data: {len(data_df)} usable questions, chance acc {(1/K).mean():.3f}")

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 85.9MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  936kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/182822 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6150 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4183 [00:00<?, ? examples/s]

(4183, 11) ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name']
cop values:
 cop
0    1348
1    1085
2     925
3     825
Name: count, dtype: int64
choice_type values:
 choice_type
single    2816
multi     1367
Name: count, dtype: int64
en usable questions: 4183
options per question:
 options
4    4183
Name: count, dtype: int64
chance accuracy (mean of 1/K): 0.250
answer position distribution:
 answer_idx
0    0.322
1    0.259
2    0.221
3    0.197
Name: proportion, dtype: float64
specialties:
 specialty
Dental                          1318
Surgery                          369
Pathology                        337
Medicine                         295
Pharmacology                     243
Pediatrics                       234
Anatomy                          234
Gynaecology & Obstetrics         224
Physiology                       171
Biochemistry                     171
Social & Preventive Medicine     129
Microbiology                   

In [11]:
# 5. Load ONE model in 4-bit (MODEL_NAME comes from the CONFIG cell). Gated models need a Hugging Face token in Colab Secrets (HF_TOKEN).
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
# from google.colab import userdata; from huggingface_hub import login; login(userdata.get("HF_TOKEN"))

MODEL_TAG = MODEL_NAME.split("/")[-1]

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb, device_map="auto")
model.eval()
print("loaded", MODEL_NAME, "| GPU memory used (GB): %.1f" % (torch.cuda.memory_allocated() / 1e9))
log_run(f"Loaded {MODEL_NAME} in 4-bit on {torch.cuda.get_device_name(0)}")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

loaded Qwen/Qwen2.5-7B-Instruct | GPU memory used (GB): 5.9
